# Exploratory data analysis

Context for *Implications of AI occupational exposure for local community
service nonprofits*. The study examines the three largest occupations in
NAICS 624200; this notebook establishes what that industry looks like, where
those three sit within it, and what the observed-use data says about them.

The task-level exposure analysis that answers the paper's sub-questions is in
`subq1.ipynb`. Everything here is plotted as published: no aggregation,
rescaling or derived categories.

| Source | Used for |
|---|---|
| BLS OEWS, May 2025 industry file | Employment and wages by occupation |
| Hatgis-Kessell et al. (2026) | Estimated hours per task per day |
| Eloundou et al. (2023) | Task exposure labels, human-rated |
| Massenkoff and McCrory (2026) | Observed AI use, by occupation and by task |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## The industry and the three occupations

In [ ]:
# ---- the sector -------------------------------------------------------------
# OEWS May 2025 industry file. NAICS 624200 is the finest industry at which BLS
# publishes occupational staffing for the population defined in the paper.
oews = pd.read_excel("data/raw/oews/oesm25in4/oesm25in4/nat4d_M2025_dl.xlsx",
                     usecols=["NAICS", "NAICS_TITLE", "OCC_CODE", "OCC_TITLE",
                              "O_GROUP", "TOT_EMP", "PCT_TOTAL", "A_MEDIAN"])
oews = oews[oews.NAICS.astype(str) == "624200"]

INDUSTRY_EMP = float(oews.loc[oews.O_GROUP == "total", "TOT_EMP"].iloc[0])
occ = oews[oews.O_GROUP == "detailed"].copy()
for c in ["TOT_EMP", "PCT_TOTAL", "A_MEDIAN"]:
    occ[c] = pd.to_numeric(occ[c], errors="coerce")

STUDY = ["21-1093", "21-1021", "11-9151"]
study = occ[occ.OCC_CODE.isin(STUDY)].set_index("OCC_CODE").loc[STUDY].reset_index()

print(f"{oews.NAICS_TITLE.iloc[0]} (NAICS 624200)")
print(f"  {len(occ)} detailed occupations, {INDUSTRY_EMP:,.0f} jobs")
print(f"  the three studied: {study.TOT_EMP.sum():,.0f} jobs "
      f"({100 * study.TOT_EMP.sum() / INDUSTRY_EMP:.1f}%)")
study[["OCC_CODE", "OCC_TITLE", "TOT_EMP", "PCT_TOTAL", "A_MEDIAN"]]

### Where the three sit

The industry is long-tailed: one occupation holds 15.7% of employment and most
hold under 1%. The three studied are its three largest.

In [ ]:
# ---- where the three sit in the industry ------------------------------------
# All 193 occupations by employment, the three studied picked out. Nothing is
# aggregated; each bar is one occupation's TOT_EMP as published.
# BLS suppresses employment for some occupations; those cannot be plotted
ranked = occ.dropna(subset=["TOT_EMP"]).sort_values("TOT_EMP", ascending=False)
print(f"{len(ranked)} of {len(occ)} occupations have published employment")
is_study = ranked.OCC_CODE.isin(STUDY).values

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.bar(range(len(ranked)), ranked.TOT_EMP, width=1.0,
       color=np.where(is_study, "#1b4d2a", "#d9dade"))
# stagger the callouts so they do not overlap
for k, (_, r) in enumerate(ranked[is_study].iterrows()):
    i = list(ranked.OCC_CODE).index(r.OCC_CODE)
    ax.annotate(f"{r.OCC_TITLE[:38]}  ({int(r.TOT_EMP):,} jobs)",
                (i, r.TOT_EMP), xytext=(26, [4, -2, -16][k]),
                textcoords="offset points", fontsize=8, color="#222",
                va="center",
                arrowprops=dict(arrowstyle="-", lw=0.6, color="#999"))
ax.set_xlabel(f"The {len(ranked)} occupations with published employment, "
              f"largest first",
              fontsize=8)
ax.set_ylabel("Jobs in the industry", fontsize=8)
ax.tick_params(labelsize=8)
ax.set_xticks([])
ax.grid(axis="y", alpha=0.25, linewidth=0.5)
ax.set_axisbelow(True)
for side in ["top", "right"]:
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

### Employment and pay

In [ ]:
# ---- employment and pay -----------------------------------------------------
# Both as published by OEWS for this industry. A_MEDIAN is the industry-specific
# median annual wage, not the national figure for the occupation.
fig, axes = plt.subplots(1, 2, figsize=(6.5, 2.4), sharey=True)
y = np.arange(len(study))[::-1]

axes[0].barh(y, study.TOT_EMP, height=0.6, color="#8899a6")
for y_, v in zip(y, study.TOT_EMP):
    axes[0].text(v + 700, y_, f"{int(v):,}", va="center", fontsize=8, color="#555")
axes[0].set_xlabel("Jobs in the industry", fontsize=8)
axes[0].set_xlim(0, study.TOT_EMP.max() * 1.3)

axes[1].barh(y, study.A_MEDIAN, height=0.6, color="#8899a6")
for y_, v in zip(y, study.A_MEDIAN):
    axes[1].text(v + 1500, y_, f"${int(v):,}", va="center", fontsize=8, color="#555")
axes[1].set_xlabel("Median annual wage, this industry", fontsize=8)
axes[1].set_xlim(0, study.A_MEDIAN.max() * 1.35)

axes[0].set_yticks(y)
axes[0].set_yticklabels(study.OCC_TITLE.str.slice(0, 40), fontsize=8)
for ax in axes:
    ax.tick_params(labelsize=8)
    ax.grid(axis="x", alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)
    for side in ["top", "right", "left"]:
        ax.spines[side].set_visible(False)
    ax.tick_params(axis="y", length=0)
plt.tight_layout()
plt.show()

## Observed AI use

Two caveats carry through the paper. Observed use is left-censored — tasks
below the reporting gate are recorded as zero rather than as missing — and it
measures one assistant, so it is a floor on AI use generally.

In [ ]:
# ---- observed AI use --------------------------------------------------------
# Massenkoff and McCrory (2026). Two files: one value per occupation, and one
# per task. A task is assigned zero unless it clears their reporting gate of
# 100 work-classified conversations, so zeros are a floor, not an absence.
use_occ = pd.read_csv("data/raw/economic_index/labor_market_impacts/job_exposure.csv")
use_task = pd.read_csv("data/raw/economic_index/labor_market_impacts/task_penetration.csv")

hours = pd.read_excel("data/raw/time_share/task_time_share_estimates.xlsx")
hours.columns = ["task_id", "task", "onet_code", "occ_name",
                 "hours", "hours_per_instance", "instances_per_day"]
hours["OCC_CODE"] = hours.onet_code.str.split(".").str[0]

elo = pd.read_csv("data/raw/gpts_are_gpts/full_labelset.tsv", sep="\t",
                  usecols=["Task ID", "Task Type", "human_exposure_agg"])

three = (hours[hours.OCC_CODE.isin(STUDY)]
         .merge(elo, left_on="task_id", right_on="Task ID", how="left")
         .merge(use_task, on="task", how="left")
         .rename(columns={"human_exposure_agg": "E", "penetration": "U"}))

print(f"{len(three)} tasks; {three.U.notna().sum()} matched to the task-level file")
print(f"tasks with any recorded use: {int((three.U > 0).sum())}\n")
print(use_occ[use_occ.occ_code.isin(STUDY)]
      .set_index("occ_code").loc[STUDY][["title", "observed_exposure"]]
      .to_string())

### Recorded use against time spent

In [ ]:
# ---- recorded use against time spent ----------------------------------------
# One dot per task. Tasks below the reporting gate sit on the zero line.
fig, axes = plt.subplots(1, 3, figsize=(6.5, 2.5), sharey=True)
for ax, (_, o) in zip(axes, study.iterrows()):
    t = three[three.OCC_CODE == o.OCC_CODE]
    used = t[t.U.fillna(0) > 0]
    ax.scatter(t.hours, t.U.fillna(0), s=22, color="#d9dade",
               edgecolor="#b9bcc2", linewidth=0.4, zorder=2)
    ax.scatter(used.hours, used.U, s=30, color="#c0392b",
               edgecolor="white", linewidth=0.5, zorder=3)
    ax.set_title("\n".join(o.OCC_TITLE[:34].split(", ")), fontsize=8, loc="left")
    ax.set_xlim(0, 3)
    ax.set_ylim(-0.05, 1.05)
    ax.tick_params(labelsize=7.5)
    ax.grid(alpha=0.25, linewidth=0.5)
    ax.set_axisbelow(True)
    for side in ["top", "right"]:
        ax.spines[side].set_visible(False)
axes[0].set_ylabel("Recorded use", fontsize=8)
axes[1].set_xlabel("Estimated hours per day for the task", fontsize=8)
plt.tight_layout()
plt.show()

print("tasks with recorded use:")
print(three[three.U.fillna(0) > 0]
      .assign(occ=lambda d: d.OCC_CODE.map(dict(zip(study.OCC_CODE, study.OCC_TITLE))))
      [["occ", "task", "hours", "E", "U"]]
      .sort_values("hours", ascending=False).round(2).to_string(index=False))

## Verification

Two relationships Hatgis-Kessell et al. (2026) report for their own data. If
they reproduce here, the join to their file is sound before anything is built
on it.

In [ ]:
# ---- verification: does the time-share data behave as published? ------------
# Two relationships Hatgis-Kessell et al. report. If these reproduce, the join
# to their file is sound. Computed over all occupations in their file, as they
# report them, not over the three studied here.
from scipy.stats import kendalltau

elo_all = pd.read_csv("data/raw/gpts_are_gpts/full_labelset.tsv", sep="\t",
                      usecols=["Task ID", "Task Type"])
national = hours.merge(elo_all, left_on="task_id", right_on="Task ID", how="inner")

# (a) Core vs Supplemental should explain almost none of the variance in hours
is_core = (national["Task Type"] == "Core").astype(float)
r2 = np.corrcoef(is_core, national.hours)[0, 1] ** 2
print(f"Core/Supplemental vs hours:  r2 = {100 * r2:.1f}%   (they report 1.3%)")

# (b) hours should correlate moderately with O*NET's own importance rating
ratings = pd.read_csv("data/raw/onet/task_ratings.csv")
importance = (ratings[ratings["Scale ID"] == "IM"][["Task ID", "Data Value"]]
              .rename(columns={"Data Value": "importance"}))
with_im = national.merge(importance, on="Task ID", how="left")

taus = []
for _, g in with_im.dropna(subset=["importance"]).groupby("OCC_CODE"):
    if len(g) < 4:
        continue
    t, p = kendalltau(g.hours, g.importance)
    if p < 0.05 and not np.isnan(t):
        taus.append(t)
print(f"hours vs O*NET importance:   mean tau = {np.mean(taus):+.2f} "
      f"across {len(taus)} occupations   (they report 0.47)")

day = hours.groupby("onet_code").hours.sum()
print(f"hours per occupation:        {day.min():.2f} to {day.max():.2f} "
      f"(fitted to a 7-hour day by construction)")

## Skills

Eloundou et al. (2023) relate exposure to O*NET's skill importance ratings in
their Table 5. The replication below is a check that their published result
reproduces; the second cell shows the ratings themselves for the three
occupations studied.

In [ ]:
# ---- skills, and Eloundou's Table 5 -----------------------------------------
# O*NET rates each occupation on 35 skills, 1-5. Eloundou et al. regress
# occupation-level exposure on the 10 Basic skills plus Programming, and report
# that science and critical thinking are negatively associated with exposure.
# That is a multivariate coefficient: pairwise, both correlate positively.
import statsmodels.api as sm

sk = pd.concat([pd.read_csv("data/raw/onet/essential_skills.csv"),
                pd.read_csv("data/raw/onet/transferable_skills.csv")])
sk = sk[sk["Scale ID"] == "IM"].copy()          # importance, not level
sk["OCC_CODE"] = sk["O*NET-SOC Code"].str.split(".").str[0]

BASIC_PLUS_PROGRAMMING = [
    "Active Listening", "Mathematics", "Reading Comprehension", "Science",
    "Speaking", "Writing", "Active Learning", "Critical Thinking",
    "Learning Strategies", "Monitoring", "Programming",
]

# their normalisation: importance rescaled from the 1-5 scale to 0-1
skills_sonet = (sk.pivot_table(index="O*NET-SOC Code", columns="Element Name",
                               values="Data Value", aggfunc="mean") - 1) / 4

published = (pd.read_csv("data/raw/gpts_are_gpts/occ_level.csv")
             .set_index("O*NET-SOC Code")
             .join(skills_sonet, how="inner")
             .dropna(subset=BASIC_PLUS_PROGRAMMING + ["human_rating_beta"]))

model = sm.OLS(published.human_rating_beta,
               sm.add_constant(published[BASIC_PLUS_PROGRAMMING])).fit()
table5 = pd.DataFrame({"coef": model.params, "se": model.bse,
                       "p": model.pvalues})
table5["sig"] = np.select([table5.p < 0.001, table5.p < 0.01, table5.p < 0.05],
                          ["***", "**", "*"], "")
table5["theirs"] = pd.Series({
    "const": -0.112, "Active Listening": 0.214, "Mathematics": 0.161,
    "Reading Comprehension": 0.470, "Science": -0.230, "Speaking": 0.133,
    "Writing": 0.467, "Active Learning": -0.065, "Critical Thinking": -0.196,
    "Learning Strategies": -0.209, "Monitoring": -0.149, "Programming": 0.623})

print(f"Eloundou et al. Table 5, replicated   n={len(published)}   "
      f"R2={model.rsquared:.3f}")
table5[["coef", "se", "sig", "theirs"]].round(3)

In [ ]:
# ---- skill importance for the three occupations -----------------------------
# O*NET's ratings as published, 1-5. No weighting or aggregation.
wide_sk = sk.pivot_table(index="OCC_CODE", columns="Element Name",
                         values="Data Value", aggfunc="mean")
SKILLS = ["Reading Comprehension", "Writing", "Active Listening", "Speaking",
          "Critical Thinking", "Science", "Programming", "Mathematics",
          "Social Perceptiveness", "Service Orientation", "Coordination",
          "Judgment and Decision Making"]

profile = (wide_sk.loc[STUDY, SKILLS].T
           .rename(columns=dict(zip(study.OCC_CODE, study.OCC_TITLE))))

fig, ax = plt.subplots(figsize=(6.5, 5.1))
y = np.arange(len(SKILLS))[::-1]
for k, (name, col) in enumerate(profile.items()):
    ax.barh(y + (1 - k) * 0.27, col, height=0.26,
            color=["#1b4d2a", "#4e9e6a", "#a8d5b8"][k], label=name[:40])
ax.set_yticks(y)
ax.set_yticklabels(SKILLS, fontsize=8)
ax.set_xlabel("Importance (O*NET, 1-5)", fontsize=8)
ax.set_xlim(0, 5)
ax.tick_params(labelsize=8)
ax.tick_params(axis="y", length=0)
ax.legend(fontsize=7.5, frameon=False, loc="upper left",
          bbox_to_anchor=(0, -0.08), ncol=1)
ax.grid(axis="x", alpha=0.25, linewidth=0.5)
ax.set_axisbelow(True)
for side in ["top", "right", "left"]:
    ax.spines[side].set_visible(False)
plt.tight_layout(rect=[0, 0.09, 1, 1])
plt.show()

profile.round(2)